# Session 39: Project Review Workshop

Data Science with Python, Level 2, Stage 6: Capstone

## Learning Objectives

By the end of this session you will be able to:

1. Review a project with checklists.
2. Refactor notebook cells into functions.
3. Test a prediction function with assert.
4. Check that predictions make sense.
5. Find where a model makes its mistakes.
6. Give and receive useful peer feedback.

## Where We Are

Your capstone is under way: a plan, data, a baseline and first models. Today is a **workshop**: short lessons followed by time to improve your own project, with a partner reviewing it.

Professional teams review every project before it is released. Reviews catch the problems that the author no longer sees: leakage, untested code, predictions that make no sense, or a model that fails for one group. You will learn four habits: refactor into functions, test with `assert`, sanity-check predictions and analyse errors. The example continues with the Student Success Predictor.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"

## 1. Review Checklists

| Area | Check |
| --- | --- |
| Question | Clear user, decision and metric written at the top |
| Data | Source stated; identifiers and leaky columns excluded |
| Model | Baseline shown; candidates compared with cross-validation |
| Evaluation | Test set used once; threshold chosen without it |
| Code | Functions with clear names; no repeated blocks; runs top to bottom |
| App | Validates input; shows units and the typical error |
| README | Pitch, result, badge, how to run, limits |

## 2. Refactoring into Functions

In [ ]:
FEATURES = ["class", "study_hours_week", "sleep_hours", "attendance_pct"]
TARGET = "passed"
SEED = 0

def load_data():
    return pd.read_csv(DATA_URL + "exam_scores.csv")

def split(df):
    return train_test_split(df[FEATURES], df[TARGET], test_size=0.2,
                            stratify=df[TARGET], random_state=SEED)

In [ ]:
def build_model(C=1.0):
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["class"]),
        ("num", StandardScaler(), FEATURES[1:]),
    ])
    return Pipeline([("prep", prep), ("model", LogisticRegression(C=C))])

exams = load_data()
X_dev, X_test, y_dev, y_test = split(exams)
model = build_model()
print("CV AUC:", cross_val_score(model, X_dev, y_dev, cv=5, scoring="roc_auc").mean().round(3))

## 3. Testing with assert

In [ ]:
model.fit(X_dev, y_dev)

def pass_probability(model, class_name, study, sleep, attendance):
    row = pd.DataFrame([{"class": class_name, "study_hours_week": study,
                         "sleep_hours": sleep, "attendance_pct": attendance}])
    return float(model.predict_proba(row)[0, 1])

p = pass_probability(model, "C", 6, 7, 88)
assert 0 <= p <= 1, "A probability must be between 0 and 1"
print("Test passed, probability", round(p, 3))

In [ ]:
for hours in [1, 5, 10, 20]:
    print(hours, "hours:", round(pass_probability(model, "C", hours, 7, 88), 2))

probs = [pass_probability(model, "C", h, 7, 88) for h in [1, 5, 10, 20]]
assert probs == sorted(probs), "More study should not lower the chance"

In [ ]:
p = pass_probability(model, "Z", 6, 7, 88)
print("Unknown class still works:", round(p, 3))

for attendance in [65, 80, 95]:
    print(attendance, round(pass_probability(model, "C", 6, 7, attendance), 2))

## 4. Where Is the Model Wrong?

In [ ]:
oof = cross_val_predict(model, X_dev, y_dev, cv=5, method="predict_proba")[:, 1]
review = X_dev.assign(failed=~y_dev, flagged=oof < 0.5)
review["study_band"] = pd.cut(review["study_hours_week"], [0, 3, 6, 10, 30])

wrong = review["failed"] != review["flagged"]
print(wrong.groupby(review["study_band"], observed=True).mean().round(2))

In [ ]:
caught = review[review["failed"]].groupby("class")["flagged"].mean()
print("Share of failing students flagged, by class:")
print(caught.round(2))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Only happy paths | Testing one normal input | Test edge cases and impossible inputs too. |
| Silent nonsense | Predictions that fall when they should rise | Write sanity checks with assert. |
| Average only | Reporting one overall score | Break errors down by group and range. |
| Vague feedback | "Looks good" or "needs work" | Name the cell, the problem and a fix. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Refactor a Messy Cell

Rewrite this repeated code as one function `pass_rate_by(df, column, bins)` and call it twice: `exams["g"] = pd.cut(exams["study_hours_week"], 4); print(exams.groupby("g", observed=True)["passed"].mean())`, and the same for `sleep_hours`.

In [ ]:
# Write your code below

### In-Class Exercise 2: Write Three Tests

For your own prediction function (or `pass_probability`), write three assert tests: the output is between 0 and 1; a clearly better input gives a higher score; the function returns a float.

In [ ]:
# Write your code below

### In-Class Exercise 3: Errors by Group

For your capstone (or the exam example), compute the error rate of out-of-fold predictions by one category and one binned numeric column. Which group is hardest?

In [ ]:
# Write your code below

### In-Class Exercise 4: Swap and Review

Swap capstones with a partner and run the three review rounds. Write the strength, the top three problems and one fix for each as comments.

In [ ]:
# Write your code below

## Summary

- Review with a checklist: question, data, model, evaluation, code, app, README.
- Refactor repeated cells into small, well-named functions.
- Put settings in named constants at the top.
- Test with `assert`: ranges, directions and edge cases.
- Break errors down by group to see where the model struggles.
- Feedback names the problem, the place and a fix.

## What's Next

The last session. You will present your capstone in five minutes with a live demo, answer questions, and look back at the whole course and ahead to what to learn next.